In [ ]:
import itertools
import time
import datetime
#scrap_store('Dinmuebles24')
from datetime import datetime
# Obtenemos la fecha de hoy y le damos formato Año-Mes-Día
fecha = datetime.today().strftime('%Y-%m-%d')
import time
import pandas as pd
from os import listdir
import requests
import re
import os
import numpy as np
import random
import gspread
import openpyxl
from openpyxl import load_workbook, Workbook
from os.path import isfile,join
from DrissionPage import ChromiumPage, ChromiumOptions
import winsound
#Aqui escribe apikey google sheets
filename=
def get_sheet_data(gsheet_name,tab_name):
    gc=gspread.service_account(filename=filename)
    sh=gc.open(gsheet_name)
    worksheet=sh.worksheet(tab_name)
    df=pd.DataFrame(worksheet.get_all_records(numericise_ignore=[4]))
    #df=pd.DataFrame(worksheet.get_all_values())
    return df
def write_df_to_sheet(gsheet_name,tab_name,df):
    gc=gspread.service_account(filename=filename)
    sh=gc.open(gsheet_name)
    worksheet=sh.worksheet(tab_name)
    set_with_dataframe(worksheet,df)
def create_sheet(gsheet_name,tab_name1):
    gc=gspread.service_account(filename=filename)
    sh=gc.open(gsheet_name)
    worksheet=sh.add_worksheet(tab_name1,rows=10000,cols=20)
def delete_sheet(gsheet_name,tab_name1):
    gc=gspread.service_account(filename=filename)
    sh=gc.open(gsheet_name)
    worksheet=sh.worksheet(tab_name1)
    sh.del_worksheet(worksheet)

In [ ]:
#En esta google sheets estan las urls de inmuebles24
gsheet_name=
tab_name =
df_ficha2=get_sheet_data(gsheet_name,tab_name2)

In [ ]:
#pagina            casa_venta
# inmuebles24      https://www.inmuebles24.com/casas-en-venta-de-...


In [ ]:
def generar_urls_desde_df(df, nombre_pagina, tipo_operacion):
    """
    Extrae la URL del dataframe y genera la lista de links con incrementos de precio.
    
    :param df: El dataframe que contiene los datos (ej. df_ficha2)
    :param nombre_pagina: El valor en la columna 'pagina' (ej. 'inmuebles24')
    :param tipo_operacion: El nombre de la columna a extraer (ej. 'casa_venta')
    """
    
    # 1. Extraemos la URL cruda directamente de tu DataFrame
    try:
        # Filtramos la fila por la página y extraemos el valor de la columna solicitada
        url_bruta = df.loc[df['pagina'] == nombre_pagina, tipo_operacion].values[0]
    except IndexError:
        return [f"Error: No se encontró la página '{nombre_pagina}' o la columna '{tipo_operacion}'."]
    
    # Verificamos si la celda está vacía o es NaN
    if pd.isna(url_bruta) or not str(url_bruta).strip():
        return ["Error: La celda está vacía o no contiene una URL."]
        
    url_bruta = str(url_bruta)

    # 2. Convertimos los números fijos de tu Google Sheet a variables dinámicas
    # Esto busca los precios fijos en tu link y los cambia por {min_precio} y {max_precio}
    if "venta" in tipo_operacion or "remates" in tipo_operacion:
        url_plantilla = url_bruta.replace('1000000', '{min_precio}').replace('2000000', '{max_precio}')
    elif "renta" in tipo_operacion:
        url_plantilla = url_bruta.replace('5000', '{min_precio}').replace('10000', '{max_precio}')
    else:
        url_plantilla = url_bruta

    lista_urls = []

    # 3. Generamos las listas según el tipo de operación
    if "{min_precio}" in url_plantilla:
        
        # --- Lógica para Ventas y Remates ---
        if "venta" in tipo_operacion or "remates" in tipo_operacion:
            # Rango inicial: 40,000 a 1,000,000
            #lista_urls.append(url_plantilla.format(min_precio=400000, max_precio=1000000))
            #lista_urls.append(url_plantilla.format(min_precio=10000000, max_precio=15000000))
            # De millón en millón (hasta 10 millones)
            for i in range(10, 20):
                min_p = i * 500000
                max_p = (i + 1) * 500000
                lista_urls.append(url_plantilla.format(min_precio=min_p, max_precio=max_p))
                
        # --- Lógica para Rentas ---
        elif "renta" in tipo_operacion:
            # De 5,000 en 5,000 (hasta 50,000)
            for i in range(1, 30):
                min_p = (i * 2000)+1
                max_p = ((i + 1) * 2000)
                lista_urls.append(url_plantilla.format(min_precio=min_p, max_precio=max_p))
                
    else:
        # Si no hay precios en el link (Traspaso, Vacacional), la dejamos estática
        lista_urls.append(url_plantilla)

    return lista_urls


# ==========================================
# CÓMO EJECUTARLO EN TU NOTEBOOK
# ==========================================

# Extraer URLs para Inmuebles24 -> Casa en Venta
urls_venta = generar_urls_desde_df(df_ficha2, 'inmuebles24', 'departamento_renta')

In [ ]:
def scroll_suave(page):
    """
    Baja la página en pequeños pasos para forzar la carga de imágenes 
    y simular comportamiento humano.
    """
    # Hacemos 4 pequeños scrolls hacia abajo
    for _ in range(4):
        # Baja 500 píxeles por vez
        page.scroll.down(500) 
        # Espera medio segundo entre cada bajada
        page.wait(0.5) 
        
    # Al final, aseguramos llegar al fondo del todo
    page.scroll.to_bottom()
    page.wait(1)



In [ ]:
CHROME_PATH = r"C:\Program Files\Google\Chrome\Application\chrome.exe"
def iniciar_navegador():
    co = ChromiumOptions().set_paths(browser_path=CHROME_PATH)
    return ChromiumPage(co)
page = iniciar_navegador()

In [ ]:
def scrap_store(lista_urls, herramienta, nombre_pagina, tipo_operacion, fecha, page):
    """
    Función de extracción actualizada con parámetros dinámicos.
    
    :param lista_urls: Lista de URLs con los rangos de precios (generada en el paso anterior).
    :param herramienta: String. Ej: 'drission' o 'selenium'.
    :param nombre_pagina: String. Ej: 'inmuebles24'.
    :param tipo_operacion: String. Ej: 'casa_venta', 'departamento_renta'.
    :param fecha: String. Ej: '2026-05-29'.
    :param page: El objeto de tu navegador (el ChromiumPage de DrissionPage).
    """
    
    # 1. GENERACIÓN DINÁMICA DEL NOMBRE DEL ARCHIVO CSV
    # Siguiendo tu regla: herramienta_pagina_operacion_fecha.csv
    csv_name = f"{herramienta}_{nombre_pagina}_{tipo_operacion}_{fecha}.csv"
    print(f"Iniciando scraping... El archivo se guardará como: {csv_name}")

    names_text = []
    urls_article = []
    imgs = []

    # 2. ITERAMOS SOBRE LA LISTA DE URLs QUE RECIBE LA FUNCIÓN
    for i, url_actual in enumerate(lista_urls):
        print(f"\n--- Procesando URL {i+1} de {len(lista_urls)} ---")
        max_retries = 3
        numero_indice = 1 # Valor por defecto por si falla la extracción del texto
        
        for attempt in range(max_retries):
            try:
                time.sleep(3)
                page.get(url_actual)
                
                try:
                    # Intenta extraer la cantidad total de propiedades
                    no_list_items = page.ele('.postingsTitle-module__title')
                    #resultado = re.search(r'([\d,]+)\s+Departamentos', no_list_items.text)
                    resultado = re.search(r'([\d,]+)', no_list_items.text)
                    if resultado:
                        numero_casas = int(resultado.group(1).replace(',', ''))
                        numero_indice = (numero_casas+29) // 30
                        print(f"Se estiman {numero_indice} páginas para esta URL.")
                    else:
                        print("No se encontró el número exacto, iterando solo la página actual.")
                except:
                    print("No se detectó el elemento del título. Se asume 1 página.")
                break # Si carga correctamente, rompe el bucle de intentos
                
            except Exception as e:
                print(f"Error cargando URL principal: {e}")
                time.sleep(5)

        # 3. ITERAMOS SOBRE LA PAGINACIÓN INTERNA DE ESA URL
        for l in range(1, numero_indice + 1): # +1 para que incluya la última página real
            print(f"Extrayendo página interna: {l}")
            
            # Reemplazo de paginación para Inmuebles24
            link = url_actual.replace('pagina-2', f'pagina-{l}')
            
            for attempt in range(max_retries):
                try:
                    page.get(link)
                    scroll_suave(page)  
                    
                    # OJO: '.ui-pdp-title' es una clase típica de MercadoLibre. Si falla Inmuebles24 aquí,
                    # cámbialo a un elemento que sepas que siempre carga en Inmuebles24.
                    page.wait.ele_displayed('.ui-pdp-title', timeout=10) 
                    
                    names = page.eles('.postingsList-module__card-container')
                    names_text1 = [name.text for name in names]
                    
                    urls_article1 = []
                    imgs1 = []

                    for articulo in names:
                        # Extraer URL
                        enlace = articulo.ele('xpath:.//a', timeout=1) 
                        url_art = enlace.attr('href') if enlace else None
                        urls_article1.append(url_art)

                        # Extraer Imagen
                        imagen = articulo.ele('xpath:.//img', timeout=1)
                        img_src = imagen.attr('src') if imagen else None
                        imgs1.append(img_src)
                    print(len(names_text1),len(urls_article1),len(imgs1))
                    # Sumamos los resultados de esta página a la lista maestra
                    names_text.extend(names_text1)
                    urls_article.extend(urls_article1)
                    imgs.extend(imgs1)
                    break # Éxito extrayendo la página interna, rompemos el retry
                    
                except Exception as e:
                    wait_time = (attempt + 1) * 5 
                    print(f"⚠️ Intento {attempt + 1} fallido en página {l}: {e}")
                    if attempt < max_retries - 1:
                        print(f"Reintentando en {wait_time} segundos...")
                        time.sleep(wait_time)
                    else:
                        print(f"❌ Se agotaron los intentos para página interna {l}.")

    # 4. GUARDADO FINAL DEL CSV
    if names_text: # Verifica que la lista no esté vacía antes de guardar
        df2 = pd.DataFrame(names_text, columns=['Text_raw'])
        df2["Url"] = urls_article
        df2["Img"] = imgs
        
        # Agregamos los parámetros como nuevas columnas por si luego unes los CSV
        df2["Pagina"] = nombre_pagina
        df2["Tipo_Operacion"] = tipo_operacion
        df2["Fecha"] = fecha
        
        df2.to_csv(csv_name, encoding='utf-8-sig', index=False) # index=False evita que se guarde la columna de números de Pandas
        print(f"\n✅ ¡Listo! Se extrajeron {len(df2)} registros. Guardado en: {csv_name}")
    else:
        print("\n⚠️ No se lograron extraer datos de ninguna de las URLs proporcionadas.")

In [ ]:
scrap_store(urls_venta, 'Drission', 'inmuebles24', 'departamento_renta', fecha, page)